In [1]:
import pandas as pd
import numpy as np

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
# Load the raw dataset with proper encoding
df = pd.read_csv('../data/raw/superstore_raw.csv', encoding='latin-1')

# Basic info
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (9994, 21)

Columns:
 ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

First 5 rows:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
print("Data Types:\n")
print(df.dtypes)
print("\n" + "="*50)
print("Missing Values:\n")
print(df.isnull().sum())
print("\n" + "="*50)
print("Duplicate Rows:", df.duplicated().sum())

Data Types:

Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

Missing Values:

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

Duplicate Rows: 0

In [4]:
# Check unique values in key columns
print("Unique Regions:", df['Region'].unique() if 'Region' in df.columns else "Column name might differ")
print("Unique Categories:", df['Category'].unique() if 'Category' in df.columns else "Check column names")

# Check for negative sales/profit
if 'Sales' in df.columns:
    print(f"\nNegative Sales: {(df['Sales'] < 0).sum()}")
if 'Profit' in df.columns:
    print(f"Negative Profit: {(df['Profit'] < 0).sum()}")

# Check date format
if 'Order Date' in df.columns:
    print(f"\nOrder Date sample: {df['Order Date'].head()}")

Unique Regions: <StringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str
Unique Categories: <StringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: str

Negative Sales: 0
Negative Profit: 1871

Order Date sample: 0     11/8/2016
1     11/8/2016
2     6/12/2016
3    10/11/2015
4    10/11/2015
Name: Order Date, dtype: str


In [5]:
# Make a copy to work on
df_clean = df.copy()

# 1. Remove duplicates
df_clean = df_clean.drop_duplicates()
print(f"After removing duplicates: {df_clean.shape}")

# 2. Handle missing values (if any)
# For Superstore dataset, usually there are none, but let's be safe
df_clean = df_clean.dropna()  # Or use fillna() depending on column
print(f"After handling missing values: {df_clean.shape}")

# 3. Convert date columns to datetime
date_cols = [col for col in df_clean.columns if 'Date' in col]
for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
    print(f"Converted {col} to datetime")

# 4. Create useful new columns
if all(col in df_clean.columns for col in ['Sales', 'Profit']):
    df_clean['Profit_Margin'] = (df_clean['Profit'] / df_clean['Sales'] * 100).round(2)
    print("Created: Profit_Margin")

if 'Order Date' in df_clean.columns:
    df_clean['Order_Year'] = df_clean['Order Date'].dt.year
    df_clean['Order_Month'] = df_clean['Order Date'].dt.month
    df_clean['Order_Quarter'] = df_clean['Order Date'].dt.quarter
    print("Created: Order_Year, Order_Month, Order_Quarter")

# 5. Remove any rows with invalid dates
df_clean = df_clean.dropna(subset=date_cols)
print(f"Final shape after cleaning: {df_clean.shape}")

After removing duplicates: (9994, 21)
After handling missing values: (9994, 21)
Converted Order Date to datetime
Converted Ship Date to datetime
Created: Profit_Margin
Created: Order_Year, Order_Month, Order_Quarter
Final shape after cleaning: (9994, 25)


In [6]:
# Save to processed folder
df_clean.to_csv('../data/processed/superstore_clean.csv', index=False)
print(" Cleaned data saved to data/processed/superstore_clean.csv")

 Cleaned data saved to data/processed/superstore_clean.csv


In [7]:
# Verify the cleaned data
print("Cleaned Data Info:")
print(df_clean.info())
print("\nSample of cleaned data:")
df_clean.head()

Cleaned Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   str           
 2   Order Date     9994 non-null   datetime64[us]
 3   Ship Date      9994 non-null   datetime64[us]
 4   Ship Mode      9994 non-null   str           
 5   Customer ID    9994 non-null   str           
 6   Customer Name  9994 non-null   str           
 7   Segment        9994 non-null   str           
 8   Country        9994 non-null   str           
 9   City           9994 non-null   str           
 10  State          9994 non-null   str           
 11  Postal Code    9994 non-null   int64         
 12  Region         9994 non-null   str           
 13  Product ID     9994 non-null   str           
 14  Category       9994 non-null   str           
 15  Sub-Category 

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Profit_Margin,Order_Year,Order_Month,Order_Quarter
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,16.00,2016,11,4
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,30.00,2016,11,4
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,47.00,2016,6,2
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,-40.00,2015,10,4
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,11.25,2015,10,4
